# Motion-Aware Object Replacement in Video — All-in-One Notebook

This notebook contains **the entire code package** (previously split into modules) in one place.

It is designed to work in two tiers:

- **Baseline mode (runs anywhere):** OpenCV-only masking (GrabCut + tracking), baseline point tracking (Farneback flow), affine motion fitting, and a **mock inpaint** (paste reference image or blur region) to validate the full pipeline.
- **Upgraded mode (best quality):** Stable Diffusion inpainting via **Diffusers** (optional). The pipeline is structured so you can later plug in SAM2 masks and CoTracker/TAPIR tracks.

---

## What you get

- A complete reproducible starter kit for motion-aware video object replacement.
- A careful critique of the original thesis approach (limitations/weaknesses) *and* a more robust system design.

> **Note:** This notebook writes files to disk using `%%writefile` so you end up with a clean Python package you can import.


## Installation

In a fresh environment:

```bash
pip install -r requirements.txt

# Optional: enable real Stable Diffusion inpainting
pip install torch diffusers transformers accelerate safetensors
```

If you plan to use OpenCV trackers like CSRT/KCF reliably, install:

```bash
pip install opencv-contrib-python
```

---


In [ ]:
from pathlib import Path

# Notebook working folder
ROOT = Path('video_replace_one_notebook')
(ROOT / 'video_replace').mkdir(parents=True, exist_ok=True)
(ROOT / 'tools').mkdir(parents=True, exist_ok=True)
print('Writing package to:', ROOT.resolve())


In [ ]:
%%writefile video_replace_one_notebook/video_replace/__init__.py
"""video_replace package.

A practical, modular starter kit for motion-aware object replacement in videos.

Key design principle:
- Provide a working baseline using OpenCV-only components.
- Allow drop-in upgrades to stronger public models (SAM2, CoTracker, Diffusers).

Nothing is trained from scratch here.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile video_replace_one_notebook/video_replace/io_utils.py
"""Video I/O utilities.

Frames are represented as numpy arrays:
- shape: (H, W, 3)
- dtype: float32
- color: RGB
- range: [0, 1]

OpenCV reads/writes in BGR uint8; we convert to/from RGB float32.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np


@dataclass(frozen=True)
class VideoData:
    frames: List[np.ndarray]
    fps: float


def _require_cv2():
    try:
        import cv2  # type: ignore
    except Exception as e:  # pragma: no cover
        raise ImportError(
            "OpenCV is required for video I/O. Install with: pip install opencv-python"
        ) from e
    return cv2


def read_video(
    path: str | Path,
    *,
    max_frames: Optional[int] = None,
    resize_to: Optional[Tuple[int, int]] = None,
) -> VideoData:
    """Read a video file into memory.

    Args:
        path: Path to a video readable by OpenCV.
        max_frames: If provided, read at most this many frames.
        resize_to: If provided, resize frames to (width, height).

    Returns:
        VideoData(frames, fps)
    """
    cv2 = _require_cv2()

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))

    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30.0

    frames: List[np.ndarray] = []
    i = 0
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if resize_to is not None:
            w, h = resize_to
            frame_bgr = cv2.resize(frame_bgr, (w, h), interpolation=cv2.INTER_AREA)

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame = frame_rgb.astype(np.float32) / 255.0
        frames.append(frame)

        i += 1
        if max_frames is not None and i >= max_frames:
            break

    cap.release()
    if len(frames) == 0:
        raise RuntimeError(f"No frames read from: {path}")

    return VideoData(frames=frames, fps=float(fps))


def write_video(
    frames: List[np.ndarray],
    fps: float,
    out_path: str | Path,
    *,
    codec: str = "mp4v",
) -> None:
    """Write frames to a video file.

    Args:
        frames: list of RGB float32 frames in [0,1]
        fps: frames per second
        out_path: output path (e.g., output.mp4)
        codec: FourCC, e.g. 'mp4v', 'avc1' (system-dependent)
    """
    cv2 = _require_cv2()

    if len(frames) == 0:
        raise ValueError("frames is empty")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(str(out_path), fourcc, float(fps), (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Failed to open video writer for: {out_path}")

    for idx, fr in enumerate(frames):
        if fr.shape[:2] != (h, w):
            raise ValueError(
                f"Frame {idx} has shape {fr.shape}, expected ({h},{w},3)."
            )
        fr_u8 = np.clip(fr * 255.0, 0, 255).astype(np.uint8)
        fr_bgr = cv2.cvtColor(fr_u8, cv2.COLOR_RGB2BGR)
        writer.write(fr_bgr)

    writer.release()


In [ ]:
%%writefile video_replace_one_notebook/video_replace/masks_sam2.py
"""Mask generation for the target object.

This module is structured so you can run in two modes:

1) Strong (optional): SAM 2 video segmentation (if you install it separately).
2) Baseline (default): OpenCV-only GrabCut + tracking.

Why this matters:
- Object replacement *fails fast* if your masks flicker.
- Many thesis prototypes rely on per-frame detectors (YOLO) and suffer from
  mask instability. A VOS / propagation approach is significantly more stable.

This repo ships with an OpenCV-only fallback so you can run end-to-end without
heavy models. When you're ready, plug in SAM2 by implementing `sam2_backend()`.
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np


def _require_cv2():
    try:
        import cv2  # type: ignore
    except Exception as e:
        raise ImportError(
            "OpenCV is required for masking fallback. Install with: pip install opencv-python"
        ) from e
    return cv2


def load_masks_from_dir(mask_dir: str | Path, expected_len: int) -> List[np.ndarray]:
    """Load masks from a directory.

    The directory should contain one mask per frame, named in sorted order.
    Supported: .png, .jpg, .jpeg.

    Masks are returned as float32 arrays with values {0.0, 1.0}.
    """
    cv2 = _require_cv2()

    mask_dir = Path(mask_dir)
    if not mask_dir.exists():
        raise FileNotFoundError(str(mask_dir))

    paths = sorted(
        [p for p in mask_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}]
    )
    if len(paths) == 0:
        raise RuntimeError(f"No mask images found in: {mask_dir}")

    if len(paths) != expected_len:
        raise ValueError(
            f"Mask count mismatch: found {len(paths)} masks, expected {expected_len}."
        )

    masks: List[np.ndarray] = []
    for p in paths:
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise RuntimeError(f"Failed to read mask: {p}")
        m = (m.astype(np.float32) / 255.0) > 0.5
        masks.append(m.astype(np.float32))

    return masks


def _make_tracker(tracker_name: str):
    cv2 = _require_cv2()
    name = tracker_name.strip().upper()

    # NOTE: availability depends on OpenCV build. We try in a sensible order.
    if name == "CSRT":
        if hasattr(cv2, "TrackerCSRT_create"):
            return cv2.TrackerCSRT_create()
    if name == "KCF":
        if hasattr(cv2, "TrackerKCF_create"):
            return cv2.TrackerKCF_create()
    if name == "MOSSE":
        if hasattr(cv2, "TrackerMOSSE_create"):
            return cv2.TrackerMOSSE_create()

    # Fallback: try legacy API
    if hasattr(cv2, "legacy"):
        legacy = cv2.legacy
        if name == "CSRT" and hasattr(legacy, "TrackerCSRT_create"):
            return legacy.TrackerCSRT_create()
        if name == "KCF" and hasattr(legacy, "TrackerKCF_create"):
            return legacy.TrackerKCF_create()
        if name == "MOSSE" and hasattr(legacy, "TrackerMOSSE_create"):
            return legacy.TrackerMOSSE_create()

    raise RuntimeError(
        f"OpenCV tracker '{tracker_name}' is not available in your OpenCV build. "
        "Try installing opencv-contrib-python, or choose another tracker."
    )


def grabcut_mask(frame_rgb: np.ndarray, rect_xywh: Tuple[int, int, int, int], iters: int = 5) -> np.ndarray:
    """Compute a foreground mask via GrabCut given a rectangle.

    Args:
        frame_rgb: HxWx3 float32 RGB in [0,1]
        rect_xywh: (x, y, w, h) rectangle
        iters: grabcut iterations

    Returns:
        HxW float32 binary mask in {0,1}
    """
    cv2 = _require_cv2()

    if frame_rgb.dtype != np.float32:
        raise ValueError("frame_rgb must be float32")

    frame_u8 = np.clip(frame_rgb * 255.0, 0, 255).astype(np.uint8)
    frame_bgr = cv2.cvtColor(frame_u8, cv2.COLOR_RGB2BGR)

    mask = np.zeros(frame_bgr.shape[:2], np.uint8)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)

    x, y, w, h = rect_xywh
    x = max(0, x)
    y = max(0, y)
    w = max(1, w)
    h = max(1, h)

    rect = (x, y, w, h)
    cv2.grabCut(frame_bgr, mask, rect, bgd, fgd, iters, cv2.GC_INIT_WITH_RECT)

    # mask values: 0 bg, 1 fg, 2 prob bg, 3 prob fg
    out = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 1.0, 0.0).astype(
        np.float32
    )

    # optional smoothing
    out = cv2.medianBlur((out * 255).astype(np.uint8), 5)
    out = (out.astype(np.float32) / 255.0) > 0.5
    return out.astype(np.float32)


def grabcut_video_masks(
    frames_rgb: List[np.ndarray],
    init_box_xywh: Tuple[int, int, int, int],
    *,
    tracker: str = "CSRT",
    grabcut_iters: int = 5,
) -> List[np.ndarray]:
    """Baseline video masks using tracking + per-frame GrabCut.

    This is not as good as SAM2/VOS, but it is:
    - reproducible
    - stable enough for demos
    - requires only OpenCV

    Args:
        frames_rgb: list of RGB float32 frames
        init_box_xywh: (x,y,w,h) on frame 0
        tracker: CSRT|KCF|MOSSE
        grabcut_iters: grabcut iterations per frame

    Returns:
        list of HxW float32 masks
    """
    cv2 = _require_cv2()

    if len(frames_rgb) == 0:
        raise ValueError("frames_rgb is empty")

    tracker_obj = _make_tracker(tracker)

    # OpenCV trackers operate on uint8 BGR
    first_u8 = np.clip(frames_rgb[0] * 255.0, 0, 255).astype(np.uint8)
    first_bgr = cv2.cvtColor(first_u8, cv2.COLOR_RGB2BGR)
    ok = tracker_obj.init(first_bgr, tuple(map(float, init_box_xywh)))
    if not ok:
        raise RuntimeError("Failed to initialize OpenCV tracker with the initial box.")

    masks: List[np.ndarray] = []

    # Frame 0 mask
    masks.append(grabcut_mask(frames_rgb[0], init_box_xywh, iters=grabcut_iters))

    for t in range(1, len(frames_rgb)):
        fr_u8 = np.clip(frames_rgb[t] * 255.0, 0, 255).astype(np.uint8)
        fr_bgr = cv2.cvtColor(fr_u8, cv2.COLOR_RGB2BGR)

        ok, box = tracker_obj.update(fr_bgr)
        if not ok:
            # If tracking fails, reuse previous mask (better than crashing)
            masks.append(masks[-1].copy())
            continue

        x, y, w, h = map(int, box)
        m = grabcut_mask(frames_rgb[t], (x, y, w, h), iters=grabcut_iters)
        masks.append(m)

    return masks


def sam2_backend(*args, **kwargs):  # pragma: no cover
    """Placeholder for SAM 2 integration.

    SAM 2 is typically installed from its official repo (not a stable pip API).
    Different forks expose different function signatures.

    Implement this function to return list[HxW float32 mask].
    """
    raise NotImplementedError(
        "SAM 2 backend is not included in this package. "
        "Use `--mask-dir` to load precomputed masks OR use the OpenCV fallback with --init-box."
    )


def sam2_video_masks(
    frames_rgb: List[np.ndarray],
    *,
    mask_dir: Optional[str | Path] = None,
    init_box_xywh: Optional[Tuple[int, int, int, int]] = None,
    prefer: str = "auto",
    tracker: str = "CSRT",
) -> List[np.ndarray]:
    """Get video masks.

    Priority:
    1) If mask_dir is provided -> load masks
    2) Else if prefer == 'sam2' -> call sam2_backend (you must implement)
    3) Else -> OpenCV GrabCut+tracking fallback (requires init_box_xywh)
    """

    if mask_dir is not None:
        return load_masks_from_dir(mask_dir, expected_len=len(frames_rgb))

    prefer = prefer.strip().lower()
    if prefer in {"sam2", "auto"}:
        if prefer == "sam2":
            return sam2_backend(frames_rgb)  # user-implemented

        # auto: try sam2 backend first, fallback to OpenCV if not available
        try:
            return sam2_backend(frames_rgb)
        except Exception:
            pass

    if init_box_xywh is None:
        raise ValueError(
            "init_box_xywh is required for the OpenCV masking fallback. "
            "Provide --init-box x,y,w,h or use --mask-dir to load masks."
        )

    return grabcut_video_masks(frames_rgb, init_box_xywh, tracker=tracker)


In [ ]:
%%writefile video_replace_one_notebook/video_replace/tracks_cotracker.py
"""Point tracking / correspondences.

Best (optional) path:
- CoTracker or TAPIR for robust point tracks in real videos.

Baseline (default):
- Sample points inside the object mask on frame 0
- Propagate points forward with dense optical flow (Farneback)

The baseline is not state-of-the-art, but it's often more stable than raw RAFT
warping for quick experiments, and it keeps the package runnable without heavy
dependencies.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np


def _require_cv2():
    try:
        import cv2  # type: ignore
    except Exception as e:
        raise ImportError(
            "OpenCV is required for baseline tracking. Install with: pip install opencv-python"
        ) from e
    return cv2


@dataclass
class Tracks:
    points: np.ndarray  # (T, N, 2) float32 (x,y)
    visibility: np.ndarray  # (T, N) bool


def sample_points_from_mask(mask: np.ndarray, n_points: int, rng: np.random.Generator) -> np.ndarray:
    """Uniformly sample points from a binary mask.

    Returns Nx2 array of (x,y).
    """
    ys, xs = np.where(mask > 0.5)
    if len(xs) == 0:
        raise ValueError("Mask is empty; cannot sample points.")

    idx = rng.choice(len(xs), size=min(n_points, len(xs)), replace=False)
    pts = np.stack([xs[idx], ys[idx]], axis=1).astype(np.float32)
    return pts


def _to_gray_u8(frame_rgb: np.ndarray) -> np.ndarray:
    cv2 = _require_cv2()
    fr_u8 = np.clip(frame_rgb * 255.0, 0, 255).astype(np.uint8)
    # OpenCV expects RGB? We'll convert to gray via cvtColor
    return cv2.cvtColor(fr_u8, cv2.COLOR_RGB2GRAY)


def _bilinear_sample_flow(flow: np.ndarray, pts_xy: np.ndarray) -> np.ndarray:
    """Sample dense flow at point locations using bilinear interpolation.

    flow: HxWx2
    pts_xy: Nx2 (x,y)
    returns Nx2 flow vectors
    """
    H, W = flow.shape[:2]
    x = pts_xy[:, 0]
    y = pts_xy[:, 1]

    x0 = np.floor(x).astype(np.int32)
    y0 = np.floor(y).astype(np.int32)
    x1 = x0 + 1
    y1 = y0 + 1

    x0 = np.clip(x0, 0, W - 1)
    x1 = np.clip(x1, 0, W - 1)
    y0 = np.clip(y0, 0, H - 1)
    y1 = np.clip(y1, 0, H - 1)

    wa = (x1 - x) * (y1 - y)
    wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y)
    wd = (x - x0) * (y - y0)

    fa = flow[y0, x0]
    fb = flow[y1, x0]
    fc = flow[y0, x1]
    fd = flow[y1, x1]

    out = (
        fa * wa[:, None]
        + fb * wb[:, None]
        + fc * wc[:, None]
        + fd * wd[:, None]
    )
    return out.astype(np.float32)


def _farneback_flow(prev_rgb: np.ndarray, cur_rgb: np.ndarray) -> np.ndarray:
    """Compute dense optical flow (baseline) with Farneback."""
    cv2 = _require_cv2()
    prev = _to_gray_u8(prev_rgb)
    cur = _to_gray_u8(cur_rgb)

    flow = cv2.calcOpticalFlowFarneback(
        prev,
        cur,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )
    return flow.astype(np.float32)


def baseline_tracks_with_flow(
    frames_rgb: List[np.ndarray],
    init_mask: np.ndarray,
    *,
    n_points: int = 500,
    seed: int = 0,
) -> Tracks:
    """Track points by advecting them with dense optical flow."""

    if len(frames_rgb) == 0:
        raise ValueError("frames_rgb is empty")

    rng = np.random.default_rng(seed)
    pts0 = sample_points_from_mask(init_mask, n_points, rng)
    N = pts0.shape[0]
    T = len(frames_rgb)

    points = np.zeros((T, N, 2), dtype=np.float32)
    vis = np.zeros((T, N), dtype=bool)

    points[0] = pts0
    vis[0] = True

    H, W = frames_rgb[0].shape[:2]

    for t in range(1, T):
        flow = _farneback_flow(frames_rgb[t - 1], frames_rgb[t])  # HxWx2
        prev_pts = points[t - 1]

        # If point was not visible, keep it as-is (or you can re-seed)
        prev_vis = vis[t - 1]
        cur_pts = prev_pts.copy()

        if np.any(prev_vis):
            flow_vecs = _bilinear_sample_flow(flow, prev_pts)
            cur_pts[prev_vis] = prev_pts[prev_vis] + flow_vecs[prev_vis]

        # Bounds check
        in_bounds = (
            (cur_pts[:, 0] >= 0)
            & (cur_pts[:, 0] <= W - 1)
            & (cur_pts[:, 1] >= 0)
            & (cur_pts[:, 1] <= H - 1)
        )

        vis[t] = prev_vis & in_bounds
        points[t] = cur_pts

    return Tracks(points=points, visibility=vis)


def cotracker_tracks(
    frames_rgb: List[np.ndarray],
    init_mask: np.ndarray,
    *,
    n_points: int = 500,
    prefer: str = "auto",
    seed: int = 0,
) -> Dict[str, np.ndarray]:
    """Compute point tracks.

    Args:
        frames_rgb: list of RGB float32 frames
        init_mask: HxW float32 mask for frame 0
        n_points: number of points to track
        prefer: 'auto'|'cotracker'|'tapir'|'baseline'
        seed: RNG seed

    Returns:
        dict with keys: 'points' (T,N,2), 'visibility' (T,N)
    """

    prefer = prefer.strip().lower()

    # Optional: users can add their CoTracker/TAPIR integration here.
    # In 'auto', we try to import a backend and fall back to baseline.
    if prefer in {"cotracker", "tapir", "auto"}:
        # Placeholder hooks
        if prefer in {"cotracker", "tapir"}:
            raise NotImplementedError(
                "Deep tracking backends (CoTracker/TAPIR) are not bundled. "
                "Use prefer='baseline' or add your backend in tracks_cotracker.py."
            )

    tr = baseline_tracks_with_flow(frames_rgb, init_mask, n_points=n_points, seed=seed)
    return {"points": tr.points, "visibility": tr.visibility}


In [ ]:
%%writefile video_replace_one_notebook/video_replace/motion_fit.py
"""Motion model fitting and warping utilities.

Given point tracks, we fit a per-frame warp W_{t-1 -> t}.

This package ships with a robust baseline: a global affine (similarity) transform
estimated with RANSAC.

You can upgrade this to a non-rigid model (TPS / piecewise affine) later.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import numpy as np


def _require_cv2():
    try:
        import cv2  # type: ignore
    except Exception as e:
        raise ImportError(
            "OpenCV is required for motion fitting. Install with: pip install opencv-python"
        ) from e
    return cv2


def fit_affine_warps(
    tracks: Dict[str, np.ndarray],
    *,
    min_points: int = 8,
) -> List[np.ndarray]:
    """Fit per-frame affine warps using tracked points.

    Args:
        tracks: dict with keys 'points' (T,N,2) and 'visibility' (T,N)
        min_points: minimum correspondences to estimate a transform

    Returns:
        list of 2x3 affine matrices A_t mapping coords from frame t -> t+1.
        Length = T-1
    """
    cv2 = _require_cv2()

    pts = tracks["points"]  # (T,N,2)
    vis = tracks["visibility"].astype(bool)  # (T,N)

    if pts.ndim != 3 or pts.shape[-1] != 2:
        raise ValueError("tracks['points'] must have shape (T,N,2)")

    T = pts.shape[0]
    warps: List[np.ndarray] = []

    for t in range(T - 1):
        v = vis[t] & vis[t + 1]
        if v.sum() < min_points:
            warps.append(np.array([[1, 0, 0], [0, 1, 0]], dtype=np.float32))
            continue

        src = pts[t, v].astype(np.float32)
        dst = pts[t + 1, v].astype(np.float32)

        # Similarity/affine partial: rotation+scale+translation (more stable).
        A, inliers = cv2.estimateAffinePartial2D(
            src,
            dst,
            method=cv2.RANSAC,
            ransacReprojThreshold=3.0,
            maxIters=2000,
            confidence=0.99,
            refineIters=10,
        )

        if A is None:
            A = np.array([[1, 0, 0], [0, 1, 0]], dtype=np.float32)
        warps.append(A.astype(np.float32))

    return warps


def fit_piecewise_affine_warps(
    tracks: Dict[str, np.ndarray],
    masks: Optional[List[np.ndarray]] = None,
    *,
    min_points: int = 8,
) -> List[np.ndarray]:
    """Compatibility wrapper.

    For now this returns a global affine per frame. The name is kept to match
    the earlier design.

    Upgrade path:
    - Build Delaunay triangulation on points and fit piecewise affine
    - Or fit TPS using thin-plate spline
    """
    _ = masks  # reserved for future upgrades
    return fit_affine_warps(tracks, min_points=min_points)


def warp_image(frame_rgb: np.ndarray, A_2x3: np.ndarray) -> np.ndarray:
    """Warp an RGB frame using an affine transform.

    frame_rgb: HxWx3 float32
    A_2x3: affine mapping from source->target
    """
    cv2 = _require_cv2()

    if frame_rgb.dtype != np.float32:
        raise ValueError("frame_rgb must be float32")

    H, W = frame_rgb.shape[:2]
    warped = cv2.warpAffine(
        frame_rgb,
        A_2x3,
        (W, H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT101,
    )
    return warped.astype(np.float32)


In [ ]:
%%writefile video_replace_one_notebook/video_replace/diffusion_inpaint.py
"""Diffusion-based inpainting (Stable Diffusion) with a temporal-friendly interface.

This module supports two modes:

1) Real diffusion inpainting via Hugging Face Diffusers (optional dependency)
   - Works with Stable Diffusion inpainting checkpoints
   - Deterministic with a fixed seed (best effort)

2) A lightweight *mock* inpainting fallback (no torch, no diffusers) so you can
   run the full pipeline end-to-end for debugging.

IMPORTANT:
- For temporal consistency, this package encourages providing a temporally-warped
  prior (previous edited frame warped into current frame). In the baseline, we
  simply composite that prior into the current frame before inpainting.
- True video consistency methods (TokenFlow, vid2vid-zero, etc.) are not bundled.
  This repo is a starter kit; you can extend this module to add those methods.
"""

from __future__ import annotations

from functools import lru_cache
from pathlib import Path
from typing import Optional

import numpy as np


def _to_pil_rgb(frame_rgb: np.ndarray):
    from PIL import Image

    fr_u8 = np.clip(frame_rgb * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(fr_u8, mode="RGB")


def _to_pil_mask(mask: np.ndarray):
    """Convert {0,1} mask to a PIL image where white=to inpaint."""
    from PIL import Image

    m_u8 = (np.clip(mask, 0, 1) * 255.0).astype(np.uint8)
    return Image.fromarray(m_u8, mode="L")


def _load_ref_image(ref_image_path: str | Path):
    from PIL import Image

    img = Image.open(ref_image_path).convert("RGB")
    return img


def _mask_bbox(mask: np.ndarray):
    ys, xs = np.where(mask > 0.5)
    if len(xs) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return x0, y0, x1, y1


def _mock_inpaint(
    image: np.ndarray,
    mask: np.ndarray,
    *,
    ref_image_path: Optional[str | Path] = None,
) -> np.ndarray:
    """A simple, non-generative fallback.

    - If ref image is provided: paste it into the mask bbox.
    - Else: blur the masked region to simulate 'content replacement'.

    This is only for pipeline debugging.
    """
    try:
        import cv2  # type: ignore
    except Exception:
        cv2 = None

    out = image.copy()
    bbox = _mask_bbox(mask)
    if bbox is None:
        return out

    x0, y0, x1, y1 = bbox
    w = max(1, x1 - x0 + 1)
    h = max(1, y1 - y0 + 1)

    if ref_image_path is not None:
        from PIL import Image

        ref = _load_ref_image(ref_image_path)
        ref = ref.resize((w, h), resample=Image.BICUBIC)
        ref_np = np.asarray(ref).astype(np.float32) / 255.0

        # Soft mask edges for nicer compositing
        m = mask[y0 : y1 + 1, x0 : x1 + 1].copy()
        if cv2 is not None:
            m = cv2.GaussianBlur(m.astype(np.float32), (0, 0), sigmaX=3)
            m = np.clip(m, 0, 1)

        out_patch = out[y0 : y1 + 1, x0 : x1 + 1]
        out[y0 : y1 + 1, x0 : x1 + 1] = (
            ref_np * m[..., None] + out_patch * (1.0 - m[..., None])
        )
        return out

    # No ref image: blur the region to indicate “edited”
    if cv2 is not None:
        region = out[y0 : y1 + 1, x0 : x1 + 1]
        blurred = cv2.GaussianBlur(region, (0, 0), sigmaX=8)
        m = mask[y0 : y1 + 1, x0 : x1 + 1]
        m = cv2.GaussianBlur(m.astype(np.float32), (0, 0), sigmaX=3)
        m = np.clip(m, 0, 1)
        out[y0 : y1 + 1, x0 : x1 + 1] = blurred * m[..., None] + region * (1 - m[..., None])
        return out

    # Pure numpy fallback: replace with mean color
    mean_color = out[mask <= 0.5].reshape(-1, 3).mean(axis=0)
    out[mask > 0.5] = mean_color
    return out


@lru_cache(maxsize=1)
def _get_inpaint_pipe(model_id: str, device: str, dtype: str):
    """Lazy-load the Diffusers inpainting pipeline."""
    import torch
    from diffusers import StableDiffusionInpaintPipeline

    torch_dtype = getattr(torch, dtype)

    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
    )
    pipe = pipe.to(device)

    # Helpful defaults
    if hasattr(pipe, "safety_checker"):
        # You can disable if you understand implications:
        # pipe.safety_checker = None
        pass

    return pipe


def inpaint_sd_temporal(
    *,
    image: np.ndarray,
    mask: np.ndarray,
    prompt: str,
    ref_image_path: Optional[str | Path] = None,
    temporal_prior: Optional[np.ndarray] = None,
    seed: int = 1234,
    model_id: str = "runwayml/stable-diffusion-inpainting",
    device: str = "cuda",
    dtype: str = "float16",
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
) -> np.ndarray:
    """Inpaint a single frame.

    Args:
        image: RGB float32 frame in [0,1]
        mask: HxW float32 mask in {0,1}; 1 means "replace / inpaint".
        prompt: text prompt describing the desired replacement
        ref_image_path: optional reference image (NOT integrated into diffusion here;
            used in mock mode only). You can extend this to IP-Adapter.
        temporal_prior: optional RGB prior (e.g., warped previous output). The
            recommended workflow is to composite prior into `image` BEFORE calling.
        seed: random seed for determinism

    Returns:
        Edited RGB float32 frame.
    """

    # The prior is currently not injected into Diffusers directly; the caller
    # should composite it into `image` before calling.
    _ = temporal_prior

    # Fast path: try Diffusers
    try:
        import torch

        pipe = _get_inpaint_pipe(model_id, device, dtype)

        img_pil = _to_pil_rgb(image)
        mask_pil = _to_pil_mask(mask)

        generator = torch.Generator(device=device).manual_seed(int(seed))

        out = pipe(
            prompt=prompt,
            image=img_pil,
            mask_image=mask_pil,
            guidance_scale=float(guidance_scale),
            num_inference_steps=int(num_inference_steps),
            generator=generator,
        )

        out_img = out.images[0]
        out_np = np.asarray(out_img).astype(np.float32) / 255.0
        return out_np

    except Exception:
        # Baseline fallback
        return _mock_inpaint(image, mask, ref_image_path=ref_image_path)


In [ ]:
%%writefile video_replace_one_notebook/video_replace/temporal_refine.py
"""Lightweight temporal refinement.

The thesis idea of optimizing latent noise with an optical-flow loss is valid but
requires a heavier, differentiable loop through the diffusion model.

This package ships with a practical, *cheap* refinement step that often reduces
flicker:
- Warp previous output into current frame
- Blend object region (and a small boundary band) with the current output

This is not a replacement for true video diffusion / attention-consistency methods,
but it is a strong improvement over per-frame independent editing.
"""

from __future__ import annotations

from typing import Optional

import numpy as np

from .motion_fit import warp_image


def _soften_mask(mask: np.ndarray, sigma: float = 3.0) -> np.ndarray:
    """Gaussian blur a mask to get soft edges."""
    try:
        import cv2  # type: ignore
    except Exception:
        # crude fallback: box blur via convolution
        k = 7
        pad = k // 2
        m = np.pad(mask, pad, mode="edge")
        out = np.zeros_like(mask, dtype=np.float32)
        for y in range(out.shape[0]):
            for x in range(out.shape[1]):
                out[y, x] = m[y : y + k, x : x + k].mean()
        return np.clip(out, 0, 1).astype(np.float32)

    m = mask.astype(np.float32)
    m = cv2.GaussianBlur(m, (0, 0), sigmaX=float(sigma))
    return np.clip(m, 0, 1).astype(np.float32)


def refine_frame_optional(
    *,
    prev_out: np.ndarray,
    cur_out: np.ndarray,
    prev_in: np.ndarray,
    cur_in: np.ndarray,
    prev_mask: np.ndarray,
    cur_mask: np.ndarray,
    warp_tminus1_to_t: np.ndarray,
    alpha: float = 0.65,
    boundary_sigma: float = 3.0,
) -> np.ndarray:
    """Reduce flicker by blending current output with warped previous output.

    Args:
        prev_out: edited frame t-1
        cur_out: edited frame t
        prev_in/cur_in: original frames (unused for now; reserved for future losses)
        prev_mask/cur_mask: masks
        warp_tminus1_to_t: affine warp mapping t-1 -> t
        alpha: how much to keep of cur_out in object region. Smaller => more temporal smoothing.
        boundary_sigma: mask edge softness
    """

    _ = (prev_in, cur_in, prev_mask)  # reserved

    warped_prev = warp_image(prev_out, warp_tminus1_to_t)

    m = _soften_mask(cur_mask, sigma=boundary_sigma)

    out = cur_out.copy()
    # Blend only inside object region (soft edges)
    out = out * (alpha * m[..., None] + (1.0 - m[..., None])) + warped_prev * ((1.0 - alpha) * m[..., None])

    return np.clip(out, 0, 1).astype(np.float32)


In [ ]:
%%writefile video_replace_one_notebook/tools/select_bbox.py
"""Interactive bounding box selection helper.

Usage:
  python tools/select_bbox.py --video input.mp4

It will show the first frame and allow you to drag a rectangle.
Then it prints: x,y,w,h

Note: This requires a GUI environment (won't work on headless servers).
"""

from __future__ import annotations

import argparse

import numpy as np


def _require_cv2():
    try:
        import cv2  # type: ignore
    except Exception as e:
        raise ImportError("pip install opencv-python") from e
    return cv2


def main():
    cv2 = _require_cv2()

    ap = argparse.ArgumentParser()
    ap.add_argument("--video", required=True)
    args = ap.parse_args()

    cap = cv2.VideoCapture(args.video)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError("Failed to read first frame")

    # Select ROI returns x,y,w,h
    r = cv2.selectROI("Select target object", frame, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = map(int, r)
    print(f"{x},{y},{w},{h}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile video_replace_one_notebook/main.py
"""Entry point for motion-aware object replacement.

This script is intentionally pragmatic:
- Works in a baseline mode with OpenCV-only dependencies (no diffusion).
- Upgrades to Stable Diffusion inpainting if you install diffusers+torch.

Typical usage (baseline masks via GrabCut+tracker):

  python main.py --video input.mp4 --out out.mp4 \
    --prompt "a wicker basket" \
    --init-box "120,220,160,140" \
    --ref ref_object.png --refine

If you already have masks for each frame:

  python main.py --video input.mp4 --out out.mp4 \
    --prompt "a wicker basket" \
    --mask-dir masks/

"""

from __future__ import annotations

import argparse
from typing import Optional, Tuple

import numpy as np

from video_replace.io_utils import read_video, write_video
from video_replace.masks_sam2 import sam2_video_masks
from video_replace.tracks_cotracker import cotracker_tracks
from video_replace.motion_fit import fit_piecewise_affine_warps, warp_image
from video_replace.diffusion_inpaint import inpaint_sd_temporal
from video_replace.temporal_refine import refine_frame_optional


def parse_box(s: str) -> Tuple[int, int, int, int]:
    parts = [p.strip() for p in s.split(",")]
    if len(parts) != 4:
        raise ValueError("--init-box must be 'x,y,w,h'")
    x, y, w, h = map(int, parts)
    return x, y, w, h


def composite(fg: np.ndarray, bg: np.ndarray, mask: np.ndarray) -> np.ndarray:
    return fg * mask[..., None] + bg * (1.0 - mask[..., None])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--video", required=True, help="Input video path")
    ap.add_argument("--out", required=True, help="Output video path")
    ap.add_argument("--prompt", required=True, help="Prompt for the replacement object")

    ap.add_argument(
        "--ref",
        default=None,
        help="Optional reference image (used in mock mode; extend for IP-Adapter)",
    )

    ap.add_argument(
        "--mask-dir",
        default=None,
        help="Directory of precomputed per-frame masks (one image per frame)",
    )
    ap.add_argument(
        "--init-box",
        default=None,
        help="Initial bbox on frame 0 in format x,y,w,h (used for OpenCV fallback)",
    )
    ap.add_argument("--tracker", default="CSRT", help="CSRT|KCF|MOSSE")

    ap.add_argument("--max-frames", type=int, default=None)
    ap.add_argument(
        "--resize",
        default=None,
        help="Optional resize 'width,height' (recommended for faster diffusion)",
    )

    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--steps", type=int, default=30)
    ap.add_argument("--guidance", type=float, default=7.5)
    ap.add_argument("--model-id", default="runwayml/stable-diffusion-inpainting")
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--dtype", default="float16")

    ap.add_argument("--refine", action="store_true", help="Enable cheap temporal refinement")

    args = ap.parse_args()

    resize_to: Optional[Tuple[int, int]] = None
    if args.resize:
        w, h = [int(x) for x in args.resize.split(",")]
        resize_to = (w, h)

    video = read_video(args.video, max_frames=args.max_frames, resize_to=resize_to)
    frames = video.frames

    init_box = parse_box(args.init_box) if args.init_box else None

    masks = sam2_video_masks(
        frames,
        mask_dir=args.mask_dir,
        init_box_xywh=init_box,
        prefer="auto",
        tracker=args.tracker,
    )

    tracks = cotracker_tracks(frames, masks[0], n_points=800, prefer="baseline", seed=args.seed)
    warps = fit_piecewise_affine_warps(tracks, masks)

    out_frames = [None] * len(frames)

    # First frame: inpaint directly
    out_frames[0] = inpaint_sd_temporal(
        image=frames[0],
        mask=masks[0],
        prompt=args.prompt,
        ref_image_path=args.ref,
        temporal_prior=None,
        seed=args.seed,
        model_id=args.model_id,
        device=args.device,
        dtype=args.dtype,
        num_inference_steps=args.steps,
        guidance_scale=args.guidance,
    )

    for t in range(1, len(frames)):
        prior = warp_image(out_frames[t - 1], warps[t - 1])
        init_img = composite(prior, frames[t], masks[t])

        cur = inpaint_sd_temporal(
            image=init_img,
            mask=masks[t],
            prompt=args.prompt,
            ref_image_path=args.ref,
            temporal_prior=prior,
            seed=args.seed,
            model_id=args.model_id,
            device=args.device,
            dtype=args.dtype,
            num_inference_steps=args.steps,
            guidance_scale=args.guidance,
        )

        if args.refine:
            cur = refine_frame_optional(
                prev_out=out_frames[t - 1],
                cur_out=cur,
                prev_in=frames[t - 1],
                cur_in=frames[t],
                prev_mask=masks[t - 1],
                cur_mask=masks[t],
                warp_tminus1_to_t=warps[t - 1],
            )

        out_frames[t] = cur

    write_video(out_frames, video.fps, args.out)
    print(f"Wrote: {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile video_replace_one_notebook/README.md
# Motion-Aware Object Replacement (Starter Package)

This is a **complete, runnable starter package** for motion-aware object replacement in videos.

It was designed as an improved implementation over a *frame-by-frame Stable Diffusion inpainting* approach by adding:
- **video-stable masks** (baseline: GrabCut + tracking; upgrade: SAM 2)
- **motion priors** (baseline: point tracks via optical flow; upgrade: CoTracker/TAPIR)
- **temporal priors** (warp previous edited frame into the next)
- **optional temporal smoothing** (cheap refinement step)

> This package is meant to be *pragmatic* and *extendable*: it runs without heavy ML dependencies, but can upgrade to Diffusers / SAM2 / advanced trackers when you install them.

---

## Quickstart (baseline mode: OpenCV-only)

1) Install dependencies:

```bash
pip install -r requirements.txt
```

2) Select the object bbox on the first frame (GUI required):

```bash
python tools/select_bbox.py --video input.mp4
```

Copy the printed `x,y,w,h`.

3) Run the pipeline:

```bash
python main.py \
  --video input.mp4 \
  --out out.mp4 \
  --prompt "a wicker basket" \
  --init-box "x,y,w,h" \
  --ref ref_object.png \
  --refine
```

**What happens in baseline mode**
- Masks are generated with GrabCut inside a tracked bbox.
- The “inpainting” step is a **mock** (it pastes the reference image into the masked region) so you can validate the full motion/mask pipeline.

---

## Enable real Stable Diffusion inpainting (Diffusers)

Install additional dependencies (example):

```bash
pip install torch diffusers transformers accelerate safetensors
```

Then run the same command. The code will automatically try to use Diffusers.

### Tips
- For speed, resize your video:

```bash
python main.py --video input.mp4 --out out.mp4 --prompt "..." --init-box "x,y,w,h" --resize "768,432"
```

- Use fewer steps to iterate quickly:

```bash
--steps 20
```

---

## Using precomputed masks

If you have masks already (one PNG per frame):

```bash
python main.py \
  --video input.mp4 \
  --out out.mp4 \
  --prompt "a robot" \
  --mask-dir masks/
```

Mask format:
- grayscale images
- white = region to replace

---

## Project layout

```
video_replace_package/
  main.py
  README.md
  requirements.txt
  tools/
    select_bbox.py
  video_replace/
    __init__.py
    io_utils.py
    masks_sam2.py
    tracks_cotracker.py
    motion_fit.py
    diffusion_inpaint.py
    temporal_refine.py
```

---

## How this improves the thesis-style approach

Instead of:
- per-frame detection/segmentation (flicker)
- per-frame diffusion (identity drift)
- raw optical flow warping (stretch artifacts)

This package does:
- **mask propagation** (stable region definition)
- **motion prior warping** of the previous edited result
- **edit constrained to a stable mask**
- optional temporal smoothing

You can then upgrade each component (SAM2, CoTracker, TokenFlow/vid2vid-zero style attention consistency) without rewriting the pipeline.

---

## License

MIT


In [ ]:
%%writefile video_replace_one_notebook/NEXT_STEPS.md
# Next steps (upgrading the baseline to SOTA)

This package is intentionally shipped with a runnable baseline.
To match the strongest research systems, upgrade modules in this order:

## 1) Masks: replace GrabCut+tracker with SAM 2 (video)
- Edit `video_replace/masks_sam2.py`
- Implement `sam2_backend(frames_rgb) -> list[masks]`
- Keep the same output contract: float32 masks in {0,1}

Why:
- Stable masks are the #1 determinant of temporal consistency.

## 2) Motion priors: replace Farneback point advection with CoTracker/TAPIR
- Edit `video_replace/tracks_cotracker.py`
- Implement a backend that returns `points` (T,N,2) and `visibility` (T,N)
- Then keep using `fit_affine_warps` or replace it with TPS/piecewise affine

## 3) True temporal diffusion editing (TokenFlow / vid2vid-zero style)
- Edit `video_replace/diffusion_inpaint.py`
- Recommended direction:
  - Run diffusion on each frame, but inject cross-frame feature/attention constraints
  - Or use a video diffusion / video inpainting model for native temporal coherence

## 4) Thesis-style latent optimization using motion consistency
If you want to implement the thesis discussion idea (optimize latent/noise to match motion):

High-level loop for frame t (pseudo):
1. Freeze the diffusion model.
2. Initialize a noise tensor eps_t (or latent z_t) near a base seed.
3. Run the inpaint sampler to produce O_t(eps_t).
4. Compute a motion loss on **background** and a warp-consistency loss on the edited object.
5. Take a few gradient steps on eps_t.
6. Decode the best eps_t.

Important: do **NOT** try to match optical flow pixelwise inside the replaced object if its
shape/appearance changes. Restrict flow loss to background or boundary bands.


In [ ]:
%%writefile video_replace_one_notebook/requirements.txt
numpy>=1.23
opencv-python>=4.7
pillow>=10.0
tqdm>=4.66


In [ ]:
%%writefile video_replace_one_notebook/LICENSE
MIT License

Copyright (c) 2026

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


## Import the package

Run the following cell after the files are written.


In [ ]:
import sys
from pathlib import Path

ROOT = Path('video_replace_one_notebook')
sys.path.insert(0, str(ROOT.resolve()))

import video_replace
print('Imported video_replace from:', video_replace.__file__)


## Quickstart (baseline, no diffusion)

This will run end-to-end even without Torch/Diffusers.

1) Choose an initial bounding box on the first frame.
2) Run the pipeline.

If you have a GUI environment, you can use:

```bash
python tools/select_bbox.py --video input.mp4
```

It prints `x,y,w,h`.

Then run (from a terminal):

```bash
python main.py --video input.mp4 --out out.mp4 --prompt "a wicker basket" --init-box "x,y,w,h" --ref ref.png --refine
```

Or run directly inside the notebook (next cell) by calling the same logic.


In [ ]:
# OPTIONAL: run inside notebook (requires input files present)
# from video_replace.io_utils import read_video, write_video
# from video_replace.masks_sam2 import sam2_video_masks
# from video_replace.tracks_cotracker import cotracker_tracks
# from video_replace.motion_fit import fit_piecewise_affine_warps, warp_image
# from video_replace.diffusion_inpaint import inpaint_sd_temporal
# from video_replace.temporal_refine import refine_frame_optional
# import numpy as np
#
# def run_in_notebook(video_path, out_path, prompt, init_box_xywh, ref=None, refine=True):
#     vid = read_video(video_path)
#     frames = vid.frames
#
#     masks = sam2_video_masks(frames, init_box_xywh=init_box_xywh, prefer='auto')
#     tracks = cotracker_tracks(frames, masks[0], n_points=800, prefer='baseline', seed=1234)
#     warps = fit_piecewise_affine_warps(tracks, masks)
#
#     out = [None]*len(frames)
#     out[0] = inpaint_sd_temporal(image=frames[0], mask=masks[0], prompt=prompt, ref_image_path=ref)
#
#     def composite(fg, bg, m):
#         return fg*m[...,None] + bg*(1-m[...,None])
#
#     for t in range(1, len(frames)):
#         prior = warp_image(out[t-1], warps[t-1])
#         init_img = composite(prior, frames[t], masks[t])
#         cur = inpaint_sd_temporal(image=init_img, mask=masks[t], prompt=prompt, ref_image_path=ref)
#         if refine:
#             cur = refine_frame_optional(
#                 prev_out=out[t-1], cur_out=cur,
#                 prev_in=frames[t-1], cur_in=frames[t],
#                 prev_mask=masks[t-1], cur_mask=masks[t],
#                 warp_tminus1_to_t=warps[t-1]
#             )
#         out[t] = cur
#
#     write_video(out, vid.fps, out_path)
#     return out_path
#
# # Example:
# # run_in_notebook('input.mp4', 'out.mp4', 'a wicker basket', (120,220,160,140), ref='ref.png', refine=True)


## Full limitations and weaknesses of the thesis work (comprehensive critique)

Below is a full critique of the thesis as written, focusing on **scientific validity**, **technical correctness**, **methodological completeness**, and **practical feasibility**.

### A. Goal-definition and problem formulation weaknesses

1) **Underspecified success criteria.** The thesis aims for “temporally consistent object replacement” and “motion inheritance,” but does not define measurable criteria for:
   - temporal stability (flicker),
   - motion correctness (trajectory + deformation + occlusion),
   - identity stability (object appearance staying constant),
   - interaction realism (shadows, contact, reflections).

2) **No formal problem statement.** It mixes multiple subproblems (segmentation, tracking, inpainting, motion transfer) without a single formal target objective.

3) **2D-only motion framing.** Motion is treated as 2D translation/scale/rotation or optical flow warping, which is insufficient for:
   - articulated or deformable objects (humans, animals, cloth),
   - camera parallax,
   - 3D interactions and occlusions.

4) **“Motion-aware” conflated with “warp-aware.”** Warping pixels is not equivalent to transferring the physical/semantic motion of an object.

---

### B. Masking/detection limitations (major failure source)

5) **Mask temporal instability is not solved.** Using per-frame detection/segmentation (YOLO) typically introduces jitter, missing parts, and inconsistent boundaries.

6) **Occlusion cases are not handled.** When an object is partially occluded, per-frame detectors often fail; this is exactly where video-consistent methods are needed.

7) **Thin structures and boundaries.** Handles, hair, ropes, legs, etc. are not reliably segmented; errors create visible inpainting artifacts.

8) **No “tracking-by-segmentation” approach.** The thesis does not adopt video object segmentation / mask propagation models that are designed for temporal stability.

---

### C. Trajectory Information Propagation weaknesses

9) **The farthest-points diameter heuristic is brittle.** It is extremely sensitive to mask noise, outliers, and small segmentation errors.

10) **Rotation is ill-defined for many shapes.** Symmetric or near-symmetric silhouettes do not provide stable rotation estimates.

11) **Scale ratio formula ambiguity.** The thesis defines size ratio using squared diameters (area scaling), but later uses it as a scaling factor. If linear scale is intended, the ratio should be \(d_2/d_1\). This is a correctness problem.

12) **Non-rigid motion is not representable.** The method cannot capture articulation or deformation (wheels spinning, limbs swinging, cloth flapping).

13) **Interaction motion is ignored.** If an object interacts with hands, surfaces, or other objects, centroid/diameter/angle is far from sufficient.

14) **Background restoration is fragile.** Saving “unmasked parts” does not solve:
   - mask boundary errors,
   - different replacement object shape,
   - disocclusion (background revealed after object moves).

---

### D. Optical flow warping weaknesses

15) **Accumulation of warping artifacts.** Iterative forward warping compounds blur, stretching, and hole artifacts over time.

16) **Flow quality depends on domain.** The thesis notes synthetic-training bias, but does not test stronger modern flow models or fine-tuning strategies.

17) **No robust occlusion strategy.** Computing occlusion masks is mentioned, but the actual editing strategy is not fully developed (e.g., forward-backward consistency filtering, disocclusion handling).

18) **Editing changes flow.** Once you replace an object, the pixel-wise optical flow inside the object region is not expected to match the original flow exactly.

19) **Camera motion separation missing.** If the camera moves, the object’s motion is entangled with global scene motion; the thesis does not explicitly model this (e.g., homography/background flow subtraction).

---

### E. Diffusion inpainting approach weaknesses

20) **Core mismatch: image model for a video problem.** Stable Diffusion inpainting operates per image. Temporal coherence is not native.

21) **Stochasticity is reduced, not eliminated.** Fixing seeds/latents can reduce variance, but doesn’t guarantee identity stability when conditions change.

22) **IP-Adapter limitation is structural.** The thesis correctly notes the missing mask alignment; this is a key reason object identity drifts.

23) **ControlNet structural conditions don’t encode motion.** Depth/canny preserve structure but are not temporal constraints.

24) **Conflicting controls not analyzed formally.** Multiple ControlNets can conflict; the thesis reports it qualitatively but doesn’t propose principled weighting/selection.

25) **Model/version specification insufficient.** For reproducibility, the exact ControlNet checkpoint, depth estimator, IP-Adapter variant, settings, and preprocessing should be stated precisely.

---

### F. Latent warping experiments weaknesses

26) **Latent space is not warp-equivariant.** Warping VAE latents with pixel flow is not theoretically justified; failure is expected.

27) **No explanation of why it fails.** The thesis reports the failure but does not provide a clear mechanism-level explanation (compressed semantics, non-local coding).

---

### G. Proposed “optical-flow loss to optimize latent” weaknesses

28) **Ill-posed if applied globally.** Matching original and edited optical flow everywhere is not correct for altered object pixels.

29) **Where to apply the loss is not defined.** A correct version must restrict loss to:
   - background regions,
   - a boundary band,
   - or coarse motion descriptors.

30) **Differentiability + compute not worked out.** Optimizing latent/noise through a diffusion sampler is expensive unless using low-step or distilled techniques.

31) **No implementation or experiments.** It remains a theoretical proposal, so claims about expected improvement are speculative.

---

### H. Experimental design and evaluation weaknesses

32) **No quantitative metrics.** There are no temporal stability or motion-consistency metrics (warp error, temporal LPIPS, etc.).

33) **No strong baselines.** Should compare against:
   - tracked copy-paste + blending,
   - classical video inpainting,
   - existing diffusion-video editing methods.

34) **Limited dataset coverage.** The videos shown are few; failure modes are not systematically tested.

35) **No ablation rigor.** Components are tried, but not in a controlled factorial/ablation framework.

36) **No runtime/compute reporting.** This is especially problematic because “carbon footprint” is a core motivation.

---

### I. Carbon footprint / “ecological” claims weaknesses

37) **No measurement.** Carbon/energy savings are asserted but not measured (GPU hours, power draw, CO₂ estimation).

38) **Inference can still be heavy.** Using pretrained diffusion still consumes significant compute, particularly for video.

39) **No explicit trade-off analysis.** Quality vs compute is not benchmarked.

---

### J. Reproducibility weaknesses

40) **Code availability is mentioned but not frozen.** A Drive link can change; reproducibility needs versioned releases.

41) **Hyperparameters not fully specified.** Steps, CFG, control weights, seeds, preprocessing sizes, etc.

42) **Data sources / licenses unclear.** Videos used are not described with licensing/permissions.

---

### K. Writing, citation, and technical consistency weaknesses

43) **YOLO version confusion.** The preliminaries describe YOLOv1-style architecture while methodology uses YOLOv8. That’s inconsistent.

44) **Some citations are informal links.** (e.g., Horn–Schunck link). Better to cite the original paper formally.

45) **Insufficient clarity on “best practices.”** The “best practices code” link is not accompanied by a minimal reproducible script.

---

### Bottom line

The thesis correctly identifies the *real* difficulty—temporal consistency—but the proposed system remains primarily a set of **image-based heuristics** and **warping experiments**. The strongest results come from controlled inpainting, but the method still lacks a **video-native consistency mechanism** and lacks **rigorous evaluation**.

---

## What a better solution needs (high-level)

1) **Video-consistent masks** (video segmentation / mask propagation).
2) **Robust motion representation** (tracks + local warps), not only global flow.
3) **A video-consistency mechanism** in diffusion (attention/feature consistency or video diffusion).
4) **A correct temporal loss** (background preservation + warp consistency + identity), not raw global flow matching.
5) **Proper evaluation** (metrics + baselines + ablations).


## Next steps

Open `NEXT_STEPS.md` in the written package for upgrade paths:

- Plug in SAM2 masks
- Plug in CoTracker / TAPIR
- Add true diffusion-video consistency (TokenFlow / vid2vid-zero)
- Add quantitative metrics and evaluation
